# Message Management

In [ ]:
from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam
from dotenv import load_dotenv, find_dotenv
import os

load_dotenv(find_dotenv())

client = OpenAI(
    api_key=os.getenv("DEEPSEEK_API_KEY"),
    base_url="https://api.deepseek.com",
)

MODEL = "deepseek-v4-flash"
MAX_TURNS = 5

messages: list[ChatCompletionMessageParam] = [
    {
        "role": "system",
        "content": "You are a rigorous research assistant. Be concise and accurate.",
    }
]


def trim_messages() -> None:
    """
    保留：
    1. system message
    2. 最近 MAX_TURNS 轮对话

    一轮对话 = user message + assistant message
    """
    global messages

    system_message = messages[0]
    conversation_messages = messages[1:]

    recent_messages = conversation_messages[-MAX_TURNS * 2:]

    messages = [system_message] + recent_messages


def chat(user_input: str) -> str:
    global messages

    messages.append({
        "role": "user",
        "content": user_input,
    })

    trim_messages()

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.2,
    )

    assistant_reply = response.choices[0].message.content or ""

    messages.append({
        "role": "assistant",
        "content": assistant_reply,
    })

    return assistant_reply


def print_messages() -> None:
    for message in messages:
        role = message.get("role")
        content = message.get("content", "")

        print(f"{role}: {content}")
        print()